# AI Cycling Coach — GPU Training
**Runtime → Change runtime type → T4 GPU** before running.

Steps:
1. Clone repo
2. Install deps
3. Generate synthetic data
4. Train
5. Save model to Google Drive + push to GitHub

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO = 'https://github.com/yossibello/ai-coach.git'

if not os.path.exists('/content/ai-coach'):
    !git clone {REPO} /content/ai-coach
else:
    !cd /content/ai-coach && git pull

%cd /content/ai-coach

import sys
sys.path.insert(0, '/content/ai-coach/backend')
os.environ['PYTHONPATH'] = '/content/ai-coach/backend'
os.environ['PYTHONIOENCODING'] = 'utf-8'

!mkdir -p ml/data backend/models

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
!pip install pandas pyarrow -q
# torch is pre-installed on Colab with CUDA support
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# ── 4. Generate synthetic data ────────────────────────────────────────────────
# ~20-40 min for 50k athletes
# Reduce to 15000 for a quick ~2h full run

ATHLETES = 50000   # change to 15000 for faster run
DATA_FILE = 'ml/data/synthetic.parquet'

if os.path.exists(DATA_FILE):
    import pandas as pd
    df = pd.read_parquet(DATA_FILE)
    print(f'Reusing existing data: {len(df):,} rows, {df.athlete_id.nunique()} athletes')
else:
    !python -m ml.training.generate_synthetic \
        --athletes {ATHLETES} \
        --output {DATA_FILE}

In [ ]:
# ── 5. Train ──────────────────────────────────────────────────────────────────
# T4 (16GB): batch 2048 fits fine, ~1-3h for 100 epochs on 50k athletes
# A100 (40GB): batch 4096, ~30min

EPOCHS     = 100
BATCH_SIZE = 2048   # safe for T4; use 4096 for A100
MODEL_FILE = 'backend/models/cycling_coach.pt'

!python -m ml.training.train \
    --data     {DATA_FILE} \
    --output   {MODEL_FILE} \
    --epochs   {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    2>&1 | tee training.log

In [ ]:
# ── 6a. Save model to Google Drive (survives session end) ─────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/ai-coach-models/'
os.makedirs(dst, exist_ok=True)
shutil.copy('backend/models/cycling_coach.pt', dst)
print('Saved to Google Drive:', dst + 'cycling_coach.pt')

In [ ]:
# ── 6b. Push model back to GitHub ─────────────────────────────────────────────
# You need a GitHub Personal Access Token (PAT) with repo write access.
# Create one at: https://github.com/settings/tokens  (Classic, repo scope)

from getpass import getpass
token = getpass('GitHub PAT (hidden): ')

!git config user.email 'colab@training'
!git config user.name 'Colab Training'
!git remote set-url origin https://{token}@github.com/yossibello/ai-coach.git
!git add backend/models/cycling_coach.pt
!git commit -m "Trained model: {ATHLETES} athletes, {EPOCHS} epochs (Colab GPU)"
!git push origin main
print('Model pushed to GitHub!')

In [ ]:
# ── 7. Quick sanity check ─────────────────────────────────────────────────────
import torch, sys
sys.path.insert(0, '/content/ai-coach/backend')
from app.ml.model import CyclingTransformer

m = CyclingTransformer()
m.load_state_dict(torch.load('backend/models/cycling_coach.pt', map_location='cpu'))
m.eval()
print('Model loaded OK')
print('Params:', sum(p.numel() for p in m.parameters()), )

# Show last training metrics
!tail -10 training.log